# microDiffusion explorer

A tiny visual lab for `microdiffusion.py`: inspect the embedded glyph world, watch forward diffusion destroy structure, train the scalar-autograd denoiser, and sample dreams from noise. The default notebook uses the tuned `x0` prediction mode because this tiny scalar MLP learns clean images more readily than invisible Gaussian residuals.

In [18]:
import importlib, math, random, time
try:
    from IPython.display import HTML, display
except ModuleNotFoundError:
    class HTML(str): pass
    def display(x): print(str(x)[:160] + ' ...')
import microdiffusion as md
md = importlib.reload(md)
random.seed(1337)

print(f'{len(md.DATA)} glyphs | {md.T} diffusion steps | pure Python scalar autograd')

16 glyphs | 20 diffusion steps | pure Python scalar autograd


In [19]:
CSS = '''
<style>
.md-wrap{display:flex;flex-wrap:wrap;gap:14px;align-items:flex-start;font-family:ui-sans-serif,system-ui,sans-serif}
.md-card{background:#111827;color:#e5e7eb;border:1px solid #374151;border-radius:8px;padding:10px;box-shadow:0 1px 2px #0003}
.md-title{font-size:12px;color:#cbd5e1;margin:0 0 8px 0;text-align:center}
.md-grid{display:grid;grid-template-columns:repeat(8,16px);grid-template-rows:repeat(8,16px);gap:1px;background:#334155;padding:2px}
.md-pix{width:16px;height:16px}
.md-note{font-size:13px;color:#334155;margin:8px 0 14px 0}
</style>
'''

def clamp01(x):
    return max(0.0, min(1.0, x))

def tile(img, title=''):
    cells = []
    for v in img:
        g = int(255 * clamp01(v))
        cells.append(f'<div class="md-pix" style="background:rgb({g},{g},{g})"></div>')
    return f'<div class="md-card"><div class="md-title">{title}</div><div class="md-grid">{"".join(cells)}</div></div>'

def gallery(items):
    display(HTML(CSS + '<div class="md-wrap">' + ''.join(tile(img, name) for name, img in items) + '</div>'))

def line_chart(values, title='loss', w=760, h=220, color='#2563eb'):
    values = list(values)
    lo, hi = min(values), max(values)
    span = hi - lo or 1.0
    pts = []
    for i, v in enumerate(values):
        x = 30 + (w - 50) * i / max(1, len(values) - 1)
        y = 20 + (h - 45) * (1 - (v - lo) / span)
        pts.append(f'{x:.1f},{y:.1f}')
    svg = f'''<svg width="{w}" height="{h}" viewBox="0 0 {w} {h}" style="background:#fff;border:1px solid #d1d5db;border-radius:8px">
    <text x="30" y="16" font-size="13" fill="#111827">{title}</text>
    <text x="30" y="{h-8}" font-size="11" fill="#6b7280">min {lo:.4f}</text>
    <text x="{w-110}" y="{h-8}" font-size="11" fill="#6b7280">max {hi:.4f}</text>
    <polyline points="{' '.join(pts)}" fill="none" stroke="{color}" stroke-width="2"/>
    </svg>'''
    display(HTML(svg))

def smooth(xs, k=25):
    return [sum(xs[max(0, i-k+1):i+1]) / len(xs[max(0, i-k+1):i+1]) for i in range(len(xs))]

## 1. The entire dataset

There is no filesystem dataset hiding offstage. These 16 glyphs are the whole training universe.

In [20]:
gallery(md.DATA)

## 2. The noise schedule

`alpha_bar_t` is the fraction of original signal left after timestep `t`.

In [21]:
line_chart(md.alpha_bars, 'cumulative signal alpha_bar_t', color='#059669')
print(['%.3f' % v for v in md.alpha_bars])

['1.000', '0.990', '0.966', '0.929', '0.880', '0.821', '0.755', '0.683', '0.608', '0.533', '0.459', '0.389', '0.325', '0.266', '0.214', '0.169', '0.132', '0.100', '0.075', '0.055', '0.040']


## 3. Forward diffusion

Pick a clean glyph, jump directly to several noise levels with the closed-form `q(x_t | x_0)`, and watch structure dissolve.

In [22]:
name, img = md.DATA[7]
x0 = [2 * v - 1 for v in img]
noise = [random.gauss(0, 1) for _ in range(64)]
frames = []
for t in [0, 3, 6, 10, 15, 20]:
    xt = x0 if t == 0 else md.q_sample(x0, t, noise)
    frames.append((f't={t}', md.pixels(xt)))
gallery(frames)

## 4. Train the denoiser

This cell uses the same `Value` graph, MLP, Adam optimizer, and DDPM sampler as the script. The tuned default is `PREDICTION = 'x0'`; switch it to `'eps'` if you want the original noise-prediction objective. Increase `STEPS` for cleaner samples; 800 is a quick visual run.

In [23]:
STEPS = 800
PREDICTION = 'x0'
random.seed(1337)
for attr in ('m', 'v'):
    if hasattr(md.adam, attr):
        delattr(md.adam, attr)

net = md.MLP([80, 32, 32, 64])
t0 = time.time()
losses = md.train(net, STEPS, verbose=True, prediction=PREDICTION)
elapsed = time.time() - t0
print(f'trained {len(net.parameters())} scalar parameters in {elapsed:.1f}s using {PREDICTION} prediction')
line_chart(smooth(losses, 25), f'{PREDICTION} training loss, 25-step moving average')

step     1/800  loss  1.0415  ###################
step    40/800  loss  1.0436  ###################
step    80/800  loss  0.9780  ####################
step   120/800  loss  0.8949  #####################
step   160/800  loss  0.7839  ######################
step   200/800  loss  0.6768  #######################
step   240/800  loss  0.6466  ########################
step   280/800  loss  0.5989  #########################
step   320/800  loss  0.5595  #########################
step   360/800  loss  0.4647  ###########################
step   400/800  loss  0.5128  ##########################
step   440/800  loss  0.4196  ############################
step   480/800  loss  0.4335  ###########################
step   520/800  loss  0.3630  #############################
step   560/800  loss  0.4298  ###########################
step   600/800  loss  0.4173  ############################
step   640/800  loss  0.3593  #############################
step   680/800  loss  0.3157  ########################

## 5. Reverse diffusion

Start from pure Gaussian noise and let the trained network predict the noise away one DDPM step at a time.

In [24]:
dream, history = md.sample(net, return_history=True, prediction=PREDICTION)
gallery([(f't={t}', img) for t, img in history])

## 6. Dream gallery

Every tile below began as noise. With more steps the model gets more decisive, but the interesting part is already visible: the samples are not random static; they inherit the little grammar of the dataset.

In [25]:
gallery([(f'dream {i+1}', md.sample(net, prediction=PREDICTION)) for i in range(12)])